# Laboratorio 26 Integración de IA y computación en la niebla
## Preparación y análisis de los datos
Familiarizarse con la carga, limpieza y análisis del dataset.

In [ ]:
import pandas as pd  

# URL del dataset de calidad de vino blanco (UCI Machine Learnin Repository)

url ='https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

# Cargar el dataset
# Este dataset utiliza ';' como separador y no tiene nombre de columnas en la primera fila.

data = pd.read_csv(url, sep=';')

# Mostrar las primeras filas del dataset    
print("Primeras lineas del dataset:")   
print(data.head())

# Mostrar información del dataset
print("\nInformación del dataset:")
print(data.info())

print("\nDescripción del dataset:")
print(data.describe())


## Limpieza y preprocesamiento de Datos.

Verificación de valores nulos y duplicados

In [ ]:
# Comprobar valores nulos
print("Valores nulos por columna: ")
print(data.isnull().sum())

# Verificar duplicados
duplicates = data.duplicated().sum()
print(f"Filas duplicadas: {duplicates}")

## Análisis exploratorio de Datos (EDA)

### Distribución de variables:
Visualizar la distribución de las variables más importantes con histogramas, por
ejemplo la acidez fija, acidez volátil, alcohol, etc. Esto permitirá entender la naturaleza
de los datos (distribuciones sesgadas, variables dominantes, etc.).

In [ ]:
import matplotlib.pyplot as plt

data.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()



## Relación con la calidad del vino:
Como la variable ‘quality’ es el objetivo principal (variable dependiente), analizar la correlación
entre esta y las variables independientes. Por ejemplo, una matriz de correlación para
identificar qué características se relacionan más fuertemente con la calidad del vino.

In [ ]:
plt.figure(figsize=(10, 8))
correlation = data.corr()
plt.imshow(correlation, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.title("Matriz de correlación", fontsize=14)
plt.show()

# Imprimir la correlación con la variable 'quality'
print("Correlación con la variable 'quality':")
print(correlation['quality'].sort_values(ascending=False))



## Creación de un modelo ligero de Machine Learning
Como la variable ‘quality’ es el objetivo principal (variable dependiente), analizar la correlación
entre esta y las variables independientes. Por ejemplo, una matriz de correlación para
identificar qué características se relacionan más fuertemente con la calidad del vino


In [ ]:
features = ['alcohol', 'pH', 'sulphates']
X = data[features]
y = data['quality']

## División de Datos de entrenamiento y prueba
Separar el dataset en conjuntos de entrenamiento (para ajustar el modelo) y prueba (para
evaluar su rendimiento en datos no vistos).

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Entrenamiento del modelo inicial de Machine learning
Entrenar un modelo de tipo Random Forest o Regresión Logística, ya que son modelos
relativamente sencillos de implementar. El objetivo es contar con un punto de partida antes de
optimizar.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
model.fit(X_train, y_train)


## Evaluación del modelo inicial

Evaluar el rendimiento del modelo utilizando métricas
adecuadas. En el caso de calidad del vino, podría emplearse el Error Cuadrático Medio
(MSE) o el Coeficiente de Determinación (R²) si se considera el problema como una
regresión. Para clasificación, se pueden usar accuracy, F1-score, etc. Aquí se asume
regresión.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE del modelo inicial: {mse}")
print(f"R² del modelo inicial: {r2}")


### Reto 1: Entrenar un modelo más simple (Regresión Lineal) comparar rendimiento

Objetivo: Comparar el Random Forest (modelo inicial) con una Regresión Lineal, en términos de MSE, R², y también en ligereza (tiempo de predicción, número de parámetros).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import time

# Entrenar el modelo de Regresión lineal
linear_model = LinearRegression()
start_train = time.time()
linear_model.fit(X_train, y_train)
end_train = time.time()

# Predicciones
start_predict = time.time()
y_predict_lr = linear_model.predict(X_test)
end_predict = time.time()

# Metricas 
mse_lr = mean_squared_error(y_test, y_predict_lr)
r2_lr = r2_score(y_test, y_predict_lr)

print("=== REgresión Lineal")
print(f"MSE: {mse_lr}")
print(f"R²: {r2_lr:.4f}")
print(f"Tiempo de entrenamiento: {end_train - start_train:.2f} segundos")
print(f"Tiempo de predicción (1 muestra): {(end_predict - start_predict)/len(X_test):.6f} segundos")
print(f"Número de parámetros: {linear_model.n_features_in_}")



### Reto 2: Aplicar una técnica de selección de características adicional (SelectKBest)

Objetivo: Ver si seleccionar las mejores características mediante una prueba estadística mejora el rendimiento o reduce la complejidad.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

# Seleccionar las 3 mejores características usando f_regression
selector = SelectKBest(score_func=f_regression, k=3)
X_new = selector.fit_transform(X, y)  # X es todo el dataset original (todas las columnas numéricas)

# Obtener nombres de las características seleccionadas
selected_features = X.columns[selector.get_support()]
print("Características seleccionadas por SelectKBest:", selected_features.tolist())

# Dividir nuevamente (usando X_new)
X_train_new, X_test_new, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

# Entrenar un modelo (por ejemplo Random Forest) con esas características
from sklearn.ensemble import RandomForestRegressor
rf_selected = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf_selected.fit(X_train_new, y_train)
y_pred_sel = rf_selected.predict(X_test_new)

print("MSE con SelectKBest:", mean_squared_error(y_test, y_pred_sel))

## Implementación base del servidor Fog (Flask)

In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import joblib  # para guardar/cargar modelo

# Suponiendo que ya tienes el modelo entrenado (por ejemplo, lr_model)
# Guarda el modelo primero:
# joblib.dump(lr_model, 'modelo_fog.pkl')

# Cargar modelo (simulamos recarga)
# model = joblib.load('modelo_fog.pkl')

# Si estás en el mismo script, usa directamente la variable lr_model
model = linear_model # o rf_model, según el que quieras usar

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = np.array(data['data']).reshape(1, -1)
    prediction = model.predict(features)
    return jsonify({'prediction': float(prediction[0])})

if __name__ == '__main__':
    app.run(debug=False, host='0.0.0.0', port=5000)

In [ ]:
from threading import Thread
import time

def run_flask():
    app.run(debug=False, port=5000, use_reloader=False)

thread = Thread(target=run_flask)
thread.start()
time.sleep(2)  # Espera a que el servidor arranque

### Reto 1: Medir el tiempo promedio de respuesta del servidor al procesar múltiples peticiones
Objetivo: Enviar 100 muestras (por ejemplo, de X_test) al endpoint /predict y calcular la latencia promedio.

In [ ]:
import requests
import time
import numpy as np

# Tomar 100 muestras de X_test (asegúrate de que X_test tenga al menos 100 filas)
muestras = X_test.iloc[:100].values.tolist()

latencies = []
for muestra in muestras:
    inicio = time.time()
    response = requests.post('http://127.0.0.1:5000/predict', json={'data': muestra})
    fin = time.time()
    latencies.append(fin - inicio)

print(f"Latencia promedio: {np.mean(latencies)*1000:.2f} ms")
print(f"Latencia máxima: {np.max(latencies)*1000:.2f} ms")
print(f"Latencia mínima: {np.min(latencies)*1000:.2f} ms")

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model_tiny = DecisionTreeRegressor(max_depth=2)
model_tiny.fit(X_train[['alcohol']], y_train)  # solo una característica

# Guardar
joblib.dump(model_tiny, 'modelo_tiny.pkl')

# Luego carga ese modelo en otra instancia del servidor (cambia el puerto, ej: 5001)